# 08.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/07-images-and-convolution/solutions.ipynb)

In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

image = np.array([[0, 0, 0, 0, 0],
                  [0, 0, 9, 9, 9],
                  [0, 0, 9, 9, 9],
                  [0, 0, 9, 9, 9],
                  [0, 0, 0, 0, 0]], dtype=float)
vertical_edge = np.array([[-1, 0, 1],
                          [-1, 0, 1],
                          [-1, 0, 1]], dtype=float)

def conv2d(img, kernel, stride=1, pad=0):
    """Slide the kernel over the image; each output = sum of (patch × kernel)."""
    img = np.pad(img, pad)
    k = kernel.shape[0]
    n_out = (img.shape[0] - k) // stride + 1
    out = np.zeros((n_out, n_out))
    for i in range(n_out):
        for j in range(n_out):
            patch = img[i * stride:i * stride + k, j * stride:j * stride + k]
            out[i, j] = (patch * kernel).sum()
    return out

Keras 3.15.1 · backend tensorflow


## Exercise 1

Each output cell = (sum of the bottom row of its 3×3 patch) − (sum of the top row of the patch).

- Output row 0 uses image rows 0–2. The top row (image row 0) is all zeros. The bottom row (image row 2) is `0 0 9 9 9`; its three windows sum to 9, 18 and 27 → **9, 18, 27**.
- Output row 1 uses image rows 1–3, which are identical → **0, 0, 0**.
- Output row 2 uses image rows 2–4: the bottom row is zeros and the top row is image row 2 → **−9, −18, −27**.

In [2]:
horizontal_edge = vertical_edge.T
out = conv2d(image, horizontal_edge)
print(out)
print("after ReLU:\n", np.maximum(out, 0))

[[  9.  18.  27.]
 [  0.   0.   0.]
 [ -9. -18. -27.]]
after ReLU:
 [[ 9. 18. 27.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]


The top edge of the square is positive (dark above, bright below) and the bottom edge is negative. ReLU keeps only the top edge. To see both, a network needs a second filter with the signs flipped; trained CNNs do learn such pairs.

## Exercise 2

1. $\lfloor (32 - 5)/1 \rfloor + 1 = 28$
2. $\lfloor (32 + 2 - 3)/2 \rfloor + 1 = \lfloor 15.5 \rfloor + 1 = 16$
3. $\lfloor (224 + 6 - 7)/2 \rfloor + 1 = \lfloor 111.5 \rfloor + 1 = 112$

In [3]:
for n, k, p, s in [(32, 5, 0, 1), (32, 3, 1, 2), (224, 7, 3, 2)]:
    x = np.zeros((1, n, n, 1))
    if p:
        x = layers.ZeroPadding2D(p)(x)
    print(n, k, p, s, "→", layers.Conv2D(1, k, strides=s)(x).shape[1:3])

32 5 0 1 → (28, 28)
32 3 1 2 → (16, 16)
224 7 3 2 → (112, 112)


## Exercise 3

In [4]:
by_hand = conv2d(image, vertical_edge, stride=2)
conv = layers.Conv2D(1, 3, strides=2, use_bias=False)
conv.build((None, 5, 5, 1))
conv.set_weights([vertical_edge.reshape(3, 3, 1, 1)])
print(by_hand)
print("Keras agrees:", np.allclose(by_hand, conv(image.reshape(1, 5, 5, 1)).numpy()[0, :, :, 0]))

[[18.  0.]
 [18.  0.]]
Keras agrees: True


Stride 2 keeps only the windows starting at columns/rows 0 and 2: output cells (0,0), (0,2), (2,0), (2,2) of the stride-1 result.

## Exercise 4

1. Each filter is 3 × 3 × 3 = 27 weights + 1 bias; 64 filters → 64 × 28 = **1,792**.
2. 5 × 5 × 3 = 75 + 1 = 76; × 64 = **4,864**.
3. 224 × 224 × 3 = 150,528 inputs; × 64 + 64 biases = **9,633,856**.

In [5]:
for name, layer, shape in [("Conv2D(64, 3)", layers.Conv2D(64, 3), (224, 224, 3)),
                           ("Conv2D(64, 5)", layers.Conv2D(64, 5), (224, 224, 3)),
                           ("Dense(64)",     layers.Dense(64),     (224 * 224 * 3,))]:
    m = keras.Sequential([keras.Input(shape), layer])
    print(f"{name:<14} {m.count_params():>10,}")

Conv2D(64, 3)       1,792
Conv2D(64, 5)       4,864
Dense(64)       9,633,856


The convolution sees the whole photo with 1,792 weights because it reuses them at every position. The dense layer needs over five thousand times as many, and it would have to learn "a boot edge at the top left" separately from "a boot edge at the bottom right".